# Cogniv Phase 1
Voice foundation: audio, ASR, intent, tools, response, and TTS.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'voice').exists():
    PROJECT_ROOT = Path(r'C:\Users\dmane\Downloads\Cogniv AI')
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('ASR checkpoint:', (PROJECT_ROOT / 'models' / 'indic-conformer-600m-int8').exists())
print('IndicF5 directory:', (PROJECT_ROOT / 'models' / 'indicf5').exists())

Project: C:\Users\dmane\Downloads\Cogniv AI
ASR checkpoint: True
IndicF5 directory: False


## Load ASR and TTS
These adapters load lazily and report exact local blockers.

In [2]:
from voice.asr import IndicConformerASR, ASRUnavailableError
from voice.tts import IndicF5TTS, TTSUnavailableError
asr = IndicConformerASR(PROJECT_ROOT / 'models' / 'indic-conformer-600m-int8')
try:
    asr.load()
    print('ASR loaded')
except ASRUnavailableError as exc:
    print('ASR unavailable:', exc)
tts = IndicF5TTS(PROJECT_ROOT / 'models' / 'indicf5', PROJECT_ROOT / 'prompts' / 'reference.wav', '')
try:
    tts.load()
    print('TTS loaded')
except TTSUnavailableError as exc:
    print('TTS unavailable:', exc)

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


c:\Users\dmane\anaconda3\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
c:\Users\dmane\anaconda3\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


ASR loaded
TTS unavailable: IndicF5 model directory not found: C:\Users\dmane\Downloads\Cogniv AI\models\indicf5


## Intent detection and tool execution

In [3]:
from voice.intent import IntentRouter
from voice.agent import generate_response
from tools.reminders import ReminderStore
from tools.memory import memory_search
router = IntentRouter()
reminders = ReminderStore()
transcript = 'Remind me to drink water at 10:30.'
intent = router.route(transcript)
tool_result = reminders.create_reminder(intent['task'], intent.get('time'))
response = generate_response(intent, tool_result)
print('USER SPEECH:', transcript)
print('TRANSCRIPT:', transcript)
print('INTENT:', intent)
print('TOOL RESULT:', tool_result)
print('COGNIV:', response)
print('MEMORY PLACEHOLDER:', memory_search('Who is my daughter?'))

USER SPEECH: Remind me to drink water at 10:30.
TRANSCRIPT: Remind me to drink water at 10:30.
INTENT: {'intent': 'create_reminder', 'task': 'drink water', 'time': '10:30'}
TOOL RESULT: {'id': 'rem_4c0148b33d204e8d8c0c6f0dc8a03d27', 'task': 'drink water', 'time': '10:30', 'date': None, 'reminder_type': None, 'recurrence': None, 'status': 'success', 'created_at': '2026-09-13T15:37:20.496336', 'success': True, 'reminder_id': 'rem_4c0148b33d204e8d8c0c6f0dc8a03d27', 'reminder': {'id': 'rem_4c0148b33d204e8d8c0c6f0dc8a03d27', 'task': 'drink water', 'time': '10:30', 'date': None, 'reminder_type': None, 'recurrence': None, 'status': 'pending', 'created_at': '2026-09-13T15:37:20.496336'}}
COGNIV: Okay. I will remind you to drink water at 10:30.
MEMORY PLACEHOLDER: {'status': 'not_connected', 'message': 'Personal memory retrieval is not connected yet.'}


## Record, transcribe, synthesize, and play
Run this section only after ASR and IndicF5 assets are available. It uses the reusable pipeline and produces a WAV.

In [4]:
from voice.pipeline import CognivVoicePipeline
pipeline = CognivVoicePipeline(asr, tts, router, reminders)
print('Use pipeline.run_microphone(input_wav, output_wav, duration=5) for the real microphone demonstration.')
print('No fake audio is generated when either model is unavailable.')

Use pipeline.run_microphone(input_wav, output_wav, duration=5) for the real microphone demonstration.
No fake audio is generated when either model is unavailable.
